# The Commuter Rush — October 2024 subway entry patterns

**Cyril James Lunzaga**

This notebook reproduces selected, source-backed comparisons from the project. Observations are **MTA estimated station-complex entries**, not unique riders, train loads, trips, or passenger origin–destination paths. All clock hours are New York local time; unobserved station-hours are **not** imputed as zero. The complete acquisition and analytical pipeline resides in `commuter_rush/` and is the canonical reproducible implementation.

## 1. Load and validate the source

The notebook intentionally reads the 310,187-row processed CSV from the local ignored `data/processed` directory. Re-acquire source data using the repository instructions if it is absent.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'commuter_rush').is_dir():
    root = root.parent
assert (root / 'commuter_rush').is_dir(), 'Open the repository or its notebooks folder in VS Code.'
source = root / 'data/processed/mta_subway_station_hour_october_2024.csv'
manifest = root / 'outputs/monthly_acquisition/source_acquisition_manifest.json'
assert source.is_file() and manifest.is_file(), 'Generate the verified October dataset first.'
evidence = json.loads(manifest.read_text(encoding='utf-8'))
assert evidence['status'] == 'month_acquisition_passed'
assert hashlib.sha256(source.read_bytes()).hexdigest() == evidence['processed_csv']['sha256']
frame = pd.read_csv(source, dtype={'station_complex_id': 'string'})
assert len(frame) == 310187
assert not frame.duplicated(['transit_timestamp', 'station_complex_id']).any()
frame['timestamp'] = pd.to_datetime(frame['transit_timestamp'], format='%Y-%m-%dT%H:%M:%S')
frame['date'] = frame['timestamp'].dt.date
frame['hour'] = frame['timestamp'].dt.hour
frame['day_type'] = frame['timestamp'].dt.weekday.map(lambda day: 'Weekday' if day < 5 else 'Weekend')
assert frame['timestamp'].nunique() == 744 and frame['date'].nunique() == 31
assert frame['station_complex_id'].nunique() == 424
assert frame['estimated_entries'].sum() == 114680137
print('Validated station-hours:', len(frame))
print('Total estimated entries:', f"{frame['estimated_entries'].sum():,.0f}")

## 2. Normalize by calendar day

There are 23 weekdays but only 8 weekend days. Comparing raw monthly weekday/weekend sums would be misleading, so we compare average daily entries.

In [ ]:
daily = frame.groupby(['date', 'day_type'], as_index=False)['estimated_entries'].sum()
means = daily.groupby('day_type')['estimated_entries'].agg(['mean', 'count'])
assert means.loc['Weekday', 'count'] == 23 and means.loc['Weekend', 'count'] == 8
ratio = means.loc['Weekday', 'mean'] / means.loc['Weekend', 'mean']
print(means.round(1))
print('Weekday / weekend mean daily entries:', round(ratio, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pd.to_datetime(daily['date']), daily['estimated_entries'] / 1e6, marker='o', linewidth=1.7)
ax.set(title='Estimated subway entries by day — October 2024', ylabel='Million estimated entries', xlabel='Local calendar date')
ax.grid(axis='y', alpha=0.2)
plt.show()

## 3. Compare the equal-length weekday rush windows

The declared morning window covers 07:00–09:59; the evening window covers 16:00–18:59. Each uses three clock hours per date. Here, percentage-point differences are calculated **within each day** before averaging across weekdays.

In [ ]:
weekdays = frame.loc[frame['day_type'].eq('Weekday')].copy()
totals = weekdays.groupby('date')['estimated_entries'].sum()
morning = weekdays.loc[weekdays['hour'].isin([7, 8, 9])].groupby('date')['estimated_entries'].sum()
evening = weekdays.loc[weekdays['hour'].isin([16, 17, 18])].groupby('date')['estimated_entries'].sum()
paired = pd.DataFrame({'morning': morning, 'evening': evening, 'total': totals}).sort_index()
assert len(paired) == 23 and not paired.isna().any().any()
paired['evening_minus_morning_pp'] = 100 * (paired['evening'] - paired['morning']) / paired['total']
print('Mean daily evening minus morning (percentage points):', round(paired['evening_minus_morning_pp'].mean(), 4))
print('Weekdays with evening > morning:', int((paired['evening'] > paired['morning']).sum()), 'of', len(paired))

## 4. Station profiles are not systemwide totals

Profiles use entry counts, not travel direction. The primary station classification has a ±0.15 balance-index threshold and >=90% observed peak-window coverage. The separate saved table also includes an alternative-window sensitivity result.

In [ ]:
station = pd.read_csv(root / 'portfolio/tables/station_peak_orientation.csv', dtype={'station_complex_id': 'string'})
assert len(station) == 424
print(station['orientation'].value_counts().to_string())
print('Station classification definitions and sensitivity: docs/statistical_evidence_protocol.md')

## 5. October 14 sensitivity

The date was preselected for an exclusion check because it has the lowest weekday entry count in this month. Excluding a date is **not** correcting the underlying source. Primary results always retain all observed dates.

In [ ]:
sensitivity = root / 'portfolio/tables/weekday_date_sensitivity.csv'
if sensitivity.is_file():
    comparison = pd.read_csv(sensitivity)
    display(comparison[['scenario', 'weekday_days', 'weekend_days', 'weekday_weekend_ratio', 'mean_evening_minus_morning_pp']])
else:
    print('Run: python -m commuter_rush.release_sensitivity')

## Interpretation and limitations

The observed October systemwide mean weekday hourly profile peaks at 17:00, but a majority of individual stations are morning-oriented under the predefined rules. Manhattan has greater evening-oriented entry shares than the other boroughs, and high-volume stations contribute disproportionately to systemwide totals. Those findings are **descriptive** and do not establish commuter origins, destinations, causality, or the same patterns in other months.

The project reports calendar-day percentile bootstrap intervals (10,000 resamples, seed 202410) as **within-month** uncertainty descriptors; days may be serially dependent. The source has 10,508 station-hours with coordinate variations, all retained in the acquisition audit. Peak eligibility concerns only the selected peak-hour windows. See the repository reports and methodology documentation for statistical choices, sensitivity, and the complete quality checks.